Import and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('../data/tracks.parquet')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Check for missing values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Valence range:', df.valence.min().round(3), 'to', df.valence.max().round(3))
print('Energy range: ', df.energy.min().round(3), 'to', df.energy.max().round(3))

Plot the mood space

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    df.valence, df.energy,
    alpha=0.08, s=4, color='#7C3AED'
)

# Label the four emotional quadrants
ax.text(0.05, 0.95, 'Angry / Tense',   transform=ax.transAxes,
        fontsize=10, color='#991B1B', va='top')
ax.text(0.75, 0.95, 'Excited / Happy',  transform=ax.transAxes,
        fontsize=10, color='#065F46', va='top')
ax.text(0.05, 0.05, 'Sad / Depressed',  transform=ax.transAxes,
        fontsize=10, color='#1E3A5F', va='bottom')
ax.text(0.75, 0.05, 'Content / Relaxed', transform=ax.transAxes,
        fontsize=10, color='#78350F', va='bottom')

ax.axvline(0.5, color='#E5E7EB', linewidth=0.8, linestyle='--')
ax.axhline(0.5, color='#E5E7EB', linewidth=0.8, linestyle='--')
ax.set_xlabel('Valence  (sad → happy)', fontsize=12)
ax.set_ylabel('Energy   (calm → intense)', fontsize=12)
ax.set_title('Track distribution in mood space', fontsize=14)
plt.tight_layout()
plt.show()

correlation heatmap of audio features

In [ ]:
# Which features are correlated with each other?
# This helps us understand if valence and energy are truly independent axes.
features = ['valence', 'energy', 'danceability',
            'tempo', 'acousticness', 'instrumentalness']

corr = df[features].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Audio feature correlations')
plt.tight_layout()
plt.show()

Sample a journey manually — quick mental test

In [ ]:
# Before building the graph, manually verify that nearby tracks
# in mood space are musically sensible neighbours.
# Pick a sad-calm track and find the 5 closest tracks manually.

from sklearn.metrics.pairwise import euclidean_distances

coords = df[['valence', 'energy']].values

# Find the track closest to sad+calm (valence=0.15, energy=0.2)
target = np.array([[0.15, 0.20]])
dists = euclidean_distances(coords, target).flatten()
closest_5 = dists.argsort()[:5]

print('Tracks closest to sad + calm mood:')
print(df.iloc[closest_5][['name', 'artist', 'valence', 'energy']].to_string())

# Find the track closest to happy+energetic (valence=0.85, energy=0.80)
target2 = np.array([[0.85, 0.80]])
dists2 = euclidean_distances(coords, target2).flatten()
closest_5b = dists2.argsort()[:5]

print()
print('Tracks closest to happy + energetic mood:')
print(df.iloc[closest_5b][['name', 'artist', 'valence', 'energy']].to_string())